# Training

> Train decentralized LeWMs on collected datasets.

In [ ]:
#| default_exp wm.train

In [ ]:
#| hide
from nbdev.showdoc import *

LeWM's objective (stable-worldmodel's `lejepa_forward`), for each agent's model on that agent's
own stream: from clips of ``history_size + num_preds`` frames (``frameskip = action_block``), predict
each next embedding from the context window, MSE + ``sigreg_weight`` x SIGReg on the embeddings.
Transitions taken after the agent terminated are left out of the prediction loss.

`train_world_model` is a plain torch loop (AdamW, linear warmup + cosine schedule, gradient
clipping, validation split) in place of stable-worldmodel's Lightning setup.

In [ ]:
#| export
from __future__ import annotations

import math
import time
from typing import Any, Callable

import torch

from stable_marl.wm.decentralized import DecentralizedWorldModel
from stable_marl.wm.loss import SIGReg

In [ ]:
#| export
def lewm_losses(model: DecentralizedWorldModel, batch: dict, sigreg: SIGReg, history_size: int, num_preds: int = 1,
                sigreg_weight: float = 0.09) -> dict[str, torch.Tensor]:
    """
    LeWM losses of a batch of clips (as returned by :class:`HDF5Dataset`, with
    ``num_steps = history_size + num_preds`` and ``frameskip = action_block``): ``loss`` (summed over
    agents, whose models share no parameters) and per-agent ``pred_loss/<a>``, ``sigreg_loss/<a>``.
    """
    frames, A, ab = batch[model.obs_key], model.num_agents, model.action_block
    B, T = frames.shape[:2]
    actions = batch['action'].reshape(B, T, ab, A, -1)                 # the action blocks leaving each frame
    terminated = batch['terminated'].reshape(B, T, A).bool() if 'terminated' in batch else None
    out, total = {}, 0.0
    for a, wm in enumerate(model.models):
        agent_actions = model.agent_actions(actions[:, :, :, a].reshape(B, T, -1))
        inputs = {wm.obs_key: frames[:, :, a], 'action': agent_actions}
        inputs.update({key: batch[key][:, :, a] for key in getattr(wm, 'extra_encoders', {})})
        enc = wm.encode(inputs)
        emb, act_emb = enc['emb'], enc['act_emb']
        pred = wm.predict(emb[:, :history_size], act_emb[:, :history_size])
        err = (pred - emb[:, num_preds:num_preds + history_size]).pow(2).mean(dim=-1)   # (B, history_size)
        if terminated is not None:   # transitions leaving a frame where the agent had terminated
            valid = (~terminated[:, :history_size, a]).float()
            pred_loss = (err * valid).sum() / valid.sum().clamp(min=1.0)
        else:
            pred_loss = err.mean()
        sigreg_loss = sigreg(emb.transpose(0, 1))
        out[f'pred_loss/{a}'], out[f'sigreg_loss/{a}'] = pred_loss, sigreg_loss
        total = total + pred_loss + sigreg_weight * sigreg_loss
    out['loss'] = total
    return out

In [ ]:
#| export
def train_world_model(
    model: DecentralizedWorldModel,
    dataset: Any,
    epochs: int = 100,
    batch_size: int = 128,
    lr: float = 5e-5,
    weight_decay: float = 1e-3,
    sigreg_weight: float = 0.09,
    sigreg_kwargs: dict | None = None,
    history_size: int | None = None,
    num_preds: int = 1,
    grad_clip: float | None = 1.0,
    warmup_fraction: float = 0.01,
    val_split: float = 0.1,
    seed: int = 0,
    device: str | torch.device = 'cpu',
    num_workers: int = 0,
    log: Callable[[dict], None] | None = print,
) -> dict[str, list[float]]:
    """
    Train the agents' LeWMs on `dataset` (an :class:`HDF5Dataset` with
    ``num_steps = history_size + num_preds``, ``frameskip = model.action_block`` and at least the
    observation, ``action`` and ``terminated`` columns, plus the models' extra inputs). Defaults: stable-worldmodel's LeWM recipe.
    Returns the mean train / validation loss of each epoch.
    """
    history_size = history_size or model.models[0].predictor.num_frames
    generator = torch.Generator().manual_seed(seed)
    n_val = int(len(dataset) * val_split)
    train_set, val_set = torch.utils.data.random_split(dataset, [len(dataset) - n_val, n_val], generator=generator)
    loader = lambda subset, shuffle: torch.utils.data.DataLoader(
        subset, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle and len(subset) > batch_size,
        num_workers=num_workers, generator=generator if shuffle else None)
    train_loader, val_loader = loader(train_set, True), loader(val_set, False)

    model.to(device)
    sigreg = SIGReg(**(sigreg_kwargs or {})).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = max(1, epochs * len(train_loader))
    warmup = max(1, int(warmup_fraction * total_steps))
    schedule = lambda step: (step + 1) / warmup if step < warmup else \
        0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total_steps - warmup)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optim, schedule)
    to_device = lambda batch: {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
    losses = lambda batch: lewm_losses(model, to_device(batch), sigreg, history_size, num_preds, sigreg_weight)

    history = {'train_loss': [], 'val_loss': []}
    for epoch in range(epochs):
        start = time.time()
        model.train()
        train_loss = []
        for batch in train_loader:
            loss = losses(batch)['loss']
            optim.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optim.step()
            scheduler.step()
            train_loss.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_loss = [losses(batch)['loss'].item() for batch in val_loader]
        history['train_loss'].append(sum(train_loss) / max(1, len(train_loss)))
        history['val_loss'].append(sum(val_loss) / len(val_loss) if val_loss else float('nan'))
        if log is not None:
            log({'epoch': epoch + 1, 'train_loss': history['train_loss'][-1], 'val_loss': history['val_loss'][-1],
                 'seconds': round(time.time() - start, 1)})
    return history

### Tests

In [ ]:
import tempfile
import stable_marl as sm

TINY = dict(image_size=32, patch_size=8, embed_dim=32, depth=2, heads=2, dim_head=16, mlp_dim=64, projector_hidden=64,
            encoder_kwargs=dict(dim=32, depth=2, heads=2, mlp_dim=64), history_size=2)
with tempfile.TemporaryDirectory() as tmp:
    world = sm.World('MultiGrid-Empty-6x6-v0', num_envs=2, agents=2, tile_size=8)
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/d.h5', episodes=4, seed=0, progress=False)
    ds = sm.HDF5Dataset(path=f'{tmp}/d.h5', num_steps=3, keys_to_load=['pov', 'action', 'terminated'])
    model = DecentralizedWorldModel.build_lewm(num_agents=2, num_actions=7, **TINY)
    torch.manual_seed(0)
    history = train_world_model(model, ds, epochs=3, batch_size=16, lr=1e-3, sigreg_kwargs={'num_proj': 64}, log=None)
    assert len(history['train_loss']) == 3 and history['train_loss'][-1] < history['train_loss'][0]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()